In [11]:
import numpy as np
import scipy

from pathlib import Path

import pathlib

from docling.document_converter import DocumentConverter
from docling_core.types.doc import DoclingDocument
from docling_core.transforms.chunker.line_chunker import LineBasedTokenChunker
from docling_core.transforms.chunker.hierarchical_chunker import HierarchicalChunker
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker

from transformers import AutoTokenizer
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

In [12]:
tokenizer = AutoTokenizer.from_pretrained('models/embedder/bge-m3')
tokenizer

XLMRobertaTokenizer(name_or_path='models/embedder/bge-m3', vocab_size=250002, model_max_length=8192, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	250001: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=False, special=True),
})

In [19]:
doc = DoclingDocument.load_from_json('export/intranet_actualites.html.json')

docling_tokenizer = HuggingFaceTokenizer(tokenizer=tokenizer, max_tokens=64)

line_chunker = LineBasedTokenChunker(tokenizer=docling_tokenizer)

chunks = line_chunker.chunk(doc)

for c in chunks:
    print(c.text)
    print('##########################')

# Actualités

5 mars 2026

## Rénovation de la cafétéria

La cafétéria sera fermée pour travaux du 23 au 27 mars 2026. Des food trucks seront présents sur le parking pendant toute la semaine.

20 février 2026


##########################
## Partenariat avec CityLoc pour la location longue durée

Veltis et le loueur CityLoc ont signé un partenariat pour proposer le Veltis Pro L en location longue durée aux commerçants. Premier objectif : 200 vélos en circulation d'ici la fin de l'année.


##########################
2 février 2026

## La nouvelle ligne de thermolaquage tourne à plein régime

Installée fin 2025, la nouvelle ligne de peinture permet de traiter deux fois plus de cadres par jour et consomme 30 % d'énergie en moins.

15 janvier 2026


##########################
## Le 1 000e Veltis Trike est sorti de l'atelier

Lancé en 2022, le Trike a franchi le cap symbolique des mille exemplaires. Il a été livré aux services techniques d'une commune voisine.
##########################


In [24]:
docling_tokenizer = HuggingFaceTokenizer(tokenizer=tokenizer, max_tokens=512)

hierarchical_chunker = HierarchicalChunker(tokenizer=docling_tokenizer)

chunks = hierarchical_chunker.chunk(doc)

for c in chunks:
    print(c.text)
    print('##########################')



5 mars 2026
##########################
La cafétéria sera fermée pour travaux du 23 au 27 mars 2026. Des food trucks seront présents sur le parking pendant toute la semaine.
##########################
20 février 2026
##########################
Veltis et le loueur CityLoc ont signé un partenariat pour proposer le Veltis Pro L en location longue durée aux commerçants. Premier objectif : 200 vélos en circulation d'ici la fin de l'année.
##########################
2 février 2026
##########################
Installée fin 2025, la nouvelle ligne de peinture permet de traiter deux fois plus de cadres par jour et consomme 30 % d'énergie en moins.
##########################
15 janvier 2026
##########################
Lancé en 2022, le Trike a franchi le cap symbolique des mille exemplaires. Il a été livré aux services techniques d'une commune voisine.
##########################


In [29]:
docling_tokenizer = HuggingFaceTokenizer(tokenizer=tokenizer, max_tokens=400)

hybrid_chunker = HybridChunker(tokenizer=docling_tokenizer)

chunks = hybrid_chunker.chunk(doc)

for c in chunks:
    print(hybrid_chunker.contextualize(c))
    print('##########################')


Actualités
5 mars 2026
##########################
Actualités
Rénovation de la cafétéria
La cafétéria sera fermée pour travaux du 23 au 27 mars 2026. Des food trucks seront présents sur le parking pendant toute la semaine.
20 février 2026
##########################
Actualités
Partenariat avec CityLoc pour la location longue durée
Veltis et le loueur CityLoc ont signé un partenariat pour proposer le Veltis Pro L en location longue durée aux commerçants. Premier objectif : 200 vélos en circulation d'ici la fin de l'année.
2 février 2026
##########################
Actualités
La nouvelle ligne de thermolaquage tourne à plein régime
Installée fin 2025, la nouvelle ligne de peinture permet de traiter deux fois plus de cadres par jour et consomme 30 % d'énergie en moins.
15 janvier 2026
##########################
Actualités
Le 1 000e Veltis Trike est sorti de l'atelier
Lancé en 2022, le Trike a franchi le cap symbolique des mille exemplaires. Il a été livré aux services techniques d'une commun

```sh
cd ~/vllm_server
source .venv/bin/activate
vllm -v
```

```sh
vllm serve ~/models/embedder/bge-m3 \
    --runner pooling \
    --served-model-name bge-m3 \
    --gpu-memory-utilization 0.15 \
    --port 8001
```

In [37]:
from openai import OpenAI

base_url = 'http://100.85.184.14:8001/v1'

client_embedder = OpenAI(
    base_url=base_url,
    api_key='no-sk'
)

In [48]:
r = client_embedder.embeddings.create(model='bge-m3', input='bien')

vectors = []
for e in r.data:
    vectors.append(np.array(e.embedding))
vectors[0]

array([-0.02014174,  0.04292379, -0.04431938, ...,  0.00202502,
       -0.03151391, -0.01025946], shape=(1024,))

In [49]:
r = client_embedder.embeddings.create(model='bge-m3', input='mal')

for e in r.data:
    vectors.append(np.array(e.embedding))
vectors

[array([-0.02014174,  0.04292379, -0.04431938, ...,  0.00202502,
        -0.03151391, -0.01025946], shape=(1024,)),
 array([ 0.01460619,  0.03760853, -0.03108895, ...,  0.02470478,
        -0.03772461,  0.01105621], shape=(1024,))]

In [50]:
from scipy.spatial import distance

distance.cosine(vectors[0], vectors[1])

np.float64(0.37486177703052537)